# Local Eval Harness v3 — Stop-on-\boxed{}, regex extraction

**Changes from v2:**
- Custom `StoppingCriteria` halts generation as soon as the most recent `\boxed{` is closed by `}` — most problems should stop well before max_new_tokens
- Scoring extracts the contents of `\boxed{...}` via regex (matches what the comp grader appears to do based on prompt format observed in smoke test)
- Per-problem progress logs (not every-10) — so we see motion
- `HOLDOUT_N=10` for first run; we'll scale up once we know the timing
- Single-sequence inference (no batching yet — v4 will batch once v3 confirms the pipeline)

**Output:** `/kaggle/working/eval_results.csv` per-row + `/kaggle/working/eval_summary.json` aggregate.

In [ ]:
# Cell 1: Config + setup
import os, random, shutil, site, stat, sys, time

import numpy as np
import torch

T0 = time.perf_counter()
def step(label):
    print(f"[{time.perf_counter() - T0:7.1f}s] {label}", flush=True)

SEED = 42
HOLDOUT_N = 10                   # small for first run
MAX_NEW_TOKENS = 1024            # cap; stop-on-boxed should fire well before this for most

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
step(f"Config: HOLDOUT_N={HOLDOUT_N}, MAX_NEW_TOKENS={MAX_NEW_TOKENS}, SEED={SEED}")

# sitedir discovery
for utility_root in ("/kaggle/input/notebooks", "/kaggle/usr/lib/notebooks", "/kaggle/lib"):
    if not os.path.isdir(utility_root):
        continue
    for root, dirs, _ in os.walk(utility_root):
        if "models" in root.split(os.sep):
            dirs[:] = []
            continue
        if os.path.basename(root) == "python_packages":
            site.addsitedir(root)

# ptxas-blackwell exec workaround (confirmed working in smoke test)
PTXAS_DST = "/tmp/ptxas-blackwell"
for src in (
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell",
    "/kaggle/input/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell",
):
    if os.path.exists(src):
        if not os.path.exists(PTXAS_DST):
            shutil.copy(src, PTXAS_DST)
            os.chmod(PTXAS_DST, stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
        os.environ["TRITON_PTXAS_PATH"] = PTXAS_DST
        os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = PTXAS_DST
        break
step("Setup done")

In [ ]:
# Cell 2: Load model + LoRA (untrained)
import kagglehub
import mamba_ssm  # noqa: F401
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
step("Tokenizer loaded")

t = time.perf_counter()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, device_map="auto", trust_remote_code=True, dtype=torch.bfloat16,
)
model.eval()
step(f"Model loaded in {time.perf_counter() - t:.0f}s")

lora_cfg = LoraConfig(
    r=32, lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_cfg)
step(f"LoRA wrapped (untrained adapter — adapter contributes 0)")

In [ ]:
# Cell 3: Helpers — \boxed{} stopping criterion + extraction
import re
from transformers import StoppingCriteria, StoppingCriteriaList

BOXED_OPEN = "\\boxed{"  # literal: \boxed{
BOXED_RE = re.compile(r"\\boxed\{([^}]*)\}")

class BoxedClosedStop(StoppingCriteria):
    """Stop when the most recent \\boxed{ in the completion has a closing }."""
    def __init__(self, tokenizer, prompt_length):
        self.tokenizer = tokenizer
        self.prompt_length = prompt_length
    def __call__(self, input_ids, scores, **kwargs):
        completion_ids = input_ids[0, self.prompt_length:]
        text = self.tokenizer.decode(completion_ids, skip_special_tokens=True)
        idx = text.rfind(BOXED_OPEN)
        if idx < 0:
            return False
        # Look for a closing } after the most recent \boxed{
        return "}" in text[idx + len(BOXED_OPEN):]

def extract_boxed(text: str) -> str:
    """Extract contents of \\boxed{...}; returns empty string if not found."""
    m = BOXED_RE.search(text)
    return m.group(1).strip() if m else ""

# Quick self-test
assert extract_boxed("reasoning... so the answer is \\boxed{42}.") == "42"
assert extract_boxed("no boxed here") == ""
step("Helpers ready")

In [ ]:
# Cell 4: Held-out validation slice (deterministic last N by id sort)
import pandas as pd

TRAIN_CSV = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
df = pd.read_csv(TRAIN_CSV).sort_values("id").reset_index(drop=True)
holdout = df.tail(HOLDOUT_N).reset_index(drop=True)
step(f"Held-out: {holdout.shape} (last {HOLDOUT_N}; first id={holdout.iloc[0]['id']}, last id={holdout.iloc[-1]['id']})")

In [ ]:
# Cell 5: Inference loop — single-sequence, stop-on-boxed, per-problem timing
results = []
t_loop = time.perf_counter()

for i, row in holdout.iterrows():
    pid = row["id"]
    prompt = row["prompt"]
    gold = str(row["answer"])

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs.input_ids.shape[1]
    stop = StoppingCriteriaList([BoxedClosedStop(tokenizer, prompt_len)])

    t_p = time.perf_counter()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            stopping_criteria=stop,
        )
    elapsed = time.perf_counter() - t_p
    n_new = out.shape[1] - prompt_len
    completion = tokenizer.decode(out[0, prompt_len:], skip_special_tokens=True)
    extracted = extract_boxed(completion)

    results.append({
        "id": pid,
        "prompt_len_tokens": int(prompt_len),
        "completion_tokens": int(n_new),
        "completion": completion,
        "extracted": extracted,
        "gold": gold,
        "correct": extracted == gold,
        "seconds": elapsed,
    })
    print(
        f"  [{i+1:2d}/{HOLDOUT_N}] id={pid} | {n_new:4d} tok in {elapsed:5.1f}s "
        f"({elapsed/max(n_new,1)*1000:.0f} ms/tok) | extracted={extracted!r:25s} | gold={gold!r:15s} | {'✓' if extracted == gold else '✗'}",
        flush=True,
    )

step(f"Inference loop done in {time.perf_counter() - t_loop:.0f}s")

In [ ]:
# Cell 6: Aggregate metrics + save
import json

n = len(results)
n_correct = sum(r["correct"] for r in results)
n_no_boxed = sum(1 for r in results if not r["extracted"])
avg_tokens = sum(r["completion_tokens"] for r in results) / max(n, 1)
avg_secs = sum(r["seconds"] for r in results) / max(n, 1)

summary = {
    "holdout_n": n,
    "seed": SEED,
    "max_new_tokens": MAX_NEW_TOKENS,
    "adapter": "untrained (= base model)",
    "accuracy_boxed": n_correct / max(n, 1),
    "correct": n_correct,
    "no_boxed_extracted": n_no_boxed,
    "avg_completion_tokens": avg_tokens,
    "avg_seconds_per_problem": avg_secs,
    "total_inference_seconds": sum(r["seconds"] for r in results),
}

out_df = pd.DataFrame(results)
out_df.to_csv("/kaggle/working/eval_results.csv", index=False)
with open("/kaggle/working/eval_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n=== Summary ===")
print(json.dumps(summary, indent=2))

In [ ]:
# Cell 7: Failure inspection — first 5 wrong predictions
wrong = [r for r in results if not r["correct"]]
print(f"\n=== Failures: {len(wrong)}/{n} ===\n")
for i, r in enumerate(wrong[:5], 1):
    print(f"--- Failure {i} (id={r['id']}) ---")
    print(f"GOLD: {r['gold']!r}")
    print(f"EXTRACTED: {r['extracted']!r}")
    print(f"COMPLETION (last 400 chars): ...{r['completion'][-400:]!r}")
    print()
step(f"Total wall time: {(time.perf_counter() - T0) / 60:.1f} min")